# V3 Colab training and validation

Mount Google Drive, install the pinned v3 dependencies, run the fixture check, and train and validate the V3 LightGBM pipeline. The V3 pipeline includes 27 features, Double Metaphone and postal code blocking, adaptive candidate depth, per-country thresholds, and a singleton guard.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

# Change this if you placed the repository elsewhere in MyDrive.
PROJECT_DIR = Path('/content/drive/MyDrive/amazon-ml')
PACKAGE = PROJECT_DIR / 'code/business_entity_resolution'
SRC = PACKAGE / 'src'
TRAIN_DIR = PROJECT_DIR / 'student_resource/dataset/train'
TEST_DIR = PROJECT_DIR / 'student_resource/dataset/test'
ARTIFACTS = PACKAGE / 'artifacts/v3_colab'
RUN_TEST_INFERENCE = False  # optional; test labels are never read

required_files = [
    *(TRAIN_DIR / name for name in ('train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv')),
    *(TEST_DIR / name for name in ('test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv')),
]
missing = [str(path) for path in required_files if not path.is_file()]
if not (SRC / 'train_v3.py').is_file():
    raise FileNotFoundError(f'V3 source code not found under {PROJECT_DIR}')
if missing:
    raise FileNotFoundError('Missing challenge files:\n' + '\n'.join(missing))

ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_DIR}')
print(f'Training data: {TRAIN_DIR}')
print(f'Test data: {TEST_DIR}')
print(f'Artifacts: {ARTIFACTS}')
print(f'Colab-visible CPU cores: {os.cpu_count() or 1}')
print(f'Free space at artifact location: {shutil.disk_usage(ARTIFACTS).free / 2**30:.1f} GiB')

## Install pinned requirements

This installs the base pipeline and packages for V3. The scripts run as child processes, so the packages are imported after installation.

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(PACKAGE / 'requirements.txt'),
    '-r', str(PACKAGE / 'requirements-models.txt'),
])
importlib.invalidate_caches()
for module in ('duckdb', 'numpy', 'lightgbm', 'rapidfuzz'):
    if importlib.util.find_spec(module) is None:
        raise ModuleNotFoundError(f'{module} is still unavailable after dependency installation')
print('Pinned dependencies are installed for V3.')

## Fixture check and V3 validation

Run the notebook from this cell through the report cell. The V3 pipeline leverages DuckDB without GPU dependencies.

In [ ]:
def run_step(label, script, *args):
    command = [sys.executable, str(script), *(str(arg) for arg in args)]
    print(f"\n=== {label} ===\n$ " + ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, command)
    print(f'=== {label} finished ===', flush=True)

run_step('V3 fixture check', SRC / 'test_v3.py')

run_step('V3 full-corpus training and held-out validation',
         SRC / 'train_v3.py', '--train-dir', TRAIN_DIR,
         '--artifacts-dir', ARTIFACTS)

run_step('V3 render validation report',
         SRC / 'report_v3.py', '--metrics', ARTIFACTS / 'training_metrics.json',
         '--output', ARTIFACTS / 'validation_report.md')

## Verify validation artifacts

Reads the generated metrics and config to display a compact summary of the V3 run.

In [ ]:
import json

metrics_path = ARTIFACTS / 'training_metrics.json'
report_path = ARTIFACTS / 'validation_report.md'
config_path = ARTIFACTS / 'model_config.json'
model_path = ARTIFACTS / 'matcher_model.joblib'

if not all(path.is_file() for path in (metrics_path, report_path, config_path, model_path)):
    raise FileNotFoundError(f'V3 is missing one or more expected outputs in {ARTIFACTS}')

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
config = json.loads(config_path.read_text(encoding='utf-8'))
validation = metrics['final_validation']

print('# V3 Model validation summary')
print(f"Macro F0.5: {validation['macro_f05']:.6f}")
print(f"Pair precision: {validation['pair_precision']:.6f}")
print(f"Pair recall: {validation['pair_recall']:.6f}")
print(f"Candidate cap/source: {config['candidate_cap_per_source']}")
print(f"Decision threshold: {config['decision_threshold']:.2f}")
print(f"Singleton guard threshold: {config.get('singleton_guard_threshold', 0.0):.2f}")
print(f"Feature count: {len(config['feature_names'])}")
print(f"Artifact directory: {ARTIFACTS}")

## Optional: generate separate test outputs

Leave `RUN_TEST_INFERENCE = False` to stop after validation. Set it to `True` in the setup cell and run this cell to generate test output. This reads test source records only; no test labels are available or used.

In [ ]:
if RUN_TEST_INFERENCE:
    run_step('V3 test inference and format validation',
             SRC / 'infer_v3.py', '--test-dir', TEST_DIR,
             '--artifacts-dir', ARTIFACTS)
else:
    print('Test inference skipped. Set RUN_TEST_INFERENCE = True to run it.')